# How the Council one-pager's numbers are made

A step-by-step walk through every figure on 5th Square's land value tax briefing for Philadelphia
City Council, from the city's own property records to the charts. It is meant for explaining and
defending the numbers in person: each section says what is being computed, why it is done that way,
and ends with the figure as the flyer prints it.

**It runs the same code that writes the flyer's numbers.** Every calculation below calls a function
in `scripts/philadelphia_council_one_pager.py` (imported here as `op`), which is what produces
`analysis/political/philadelphia_council_one_pager/numbers.json`. The notebook adds explanation and
charts, not a second implementation, and the last section checks its results against that file.

**The reform in one paragraph.** Philadelphia taxes land and buildings at one rate, 1.3998% (City plus
School District). A land value tax splits the rate: land is taxed at 4 times the building rate, and
the two rates are set so the City and School District collect exactly what they collect today.
Nobody's total assessment changes. What changes is how each total is divided between land and
building, because the Office of Property Assessment's land values are replaced by estimates from
land sales, and then which rate each part pays. A property pays less when its land is a smaller share
of its value than the city's land is of all taxable value.

| Section | Flyer figures |
|---|---|
| 2. Tax parameters | today's rate and the Homestead Exemption |
| 5. The rates | the land and building rates, and the "about a third" rule of thumb |
| 7. Who saves | the share of homes paying less, the median saving, homestead homes paying less and more |
| 8. The rowhouse example | the owner's rowhouse, the same house rented, the empty lot next door |
| 9. Neighborhoods | the income and non-white-share bars, "8 in 10" and "9 in 10", the FHFA check |
| 10. Property types | the share paying less or more for homes, apartments, commercial and vacant lots |
| 11. Idle land | taxable vacant and parking acres and lots, the square on the map, the share paying more |
| 12. The Homestead Exemption under current law | the share of exempt homeowners who would pay more |
| 13. Abatements | never pay more / pay more for a time / pay more for good, and after expiry |
| 14. The largest empty tracts | the range for homes' total saving |

The figures themselves appear only in the outputs, never in this text, so the explanation stays
right when the data is rebuilt.

Run it top to bottom. It takes a few minutes, almost all of it in loading and classifying the parcel
roll and in re-solving the rates under alternatives.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lvt" / "philadelphia.py").exists())
sys.path[:0] = [str(REPO), str(REPO / "scripts")]

import philadelphia_council_one_pager as op
from lvt.philadelphia import HOMESTEAD_ORDERS, LARGE_TRACT_TREATMENTS, _decompose_exemptions, tax_year_params

ORDER = "value_share"          # the Homestead Exemption rule the flyer uses (section 6)
PURPLE, AMBER, RED, INK, MUTED = "#5904FE", "#FFC100", "#D8432F", "#060606", "#4E504A"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10})
pd.options.display.float_format = "{:,.2f}".format
print("repository:", REPO)

## 1. Where the data comes from

Everything starts from public records. The table below lists each input this notebook reads, whether
it is on disk, and how it was made. Rebuilding them from scratch is slow (the land-value model alone
takes hours), so the notebook reads the cached copies; the cells after the table show the commands
that regenerate each one.

| Input | What it is | Where it comes from |
|---|---|---|
| `cities/philadelphia/data/parcels_ty2026.gpq` | the tax year 2026 roll, one row per parcel | OPA via the City's Carto API (`phl.carto.com/api/v2/sql`): the `assessments` table for 2026 (taxable and exempt land and building) joined to `opa_properties_public` (category, lot area, owner, Homestead Exemption, location) |
| `analysis/data/philadelphia_lycd_reassessment_ty2026_s5.csv` | each parcel's land value from land sales (S5), with its block group's income and non-white share | the land-value model in the separate `philly_open_avmkit` repository, painted onto parcels by `cities/philadelphia/model_lycd_reassessment.ipynb`; demographics from the Census ACS 5-year 2022 at block-group level (median household income `B19013`, race and ethnicity `B03002`) |
| `analysis/data/philadelphia_lycd_reassessment_ty2026_s5_support.json` | the lot size past which the land sales stop testing S5 | the same model, written with the export |
| `cities/philadelphia/data/vacant_land_lni.gpq` | vacant land | L&I's Vacant Property Indicators (land) ArcGIS layer |
| `cities/philadelphia/data/surface_parking_opa.gpq` | surface parking lots | OPA parcels whose building code says `PKG LOT` |
| `analysis/ownership/philadelphia/opa_attributes.parquet` | building codes and zoning | OPA `opa_properties_public` |
| `analysis/ownership/philadelphia/opa_mailing.parquet` | owners' mailing addresses (to spot owner-occupants) | OPA `opa_properties_public` |
| `cities/philadelphia/data/fhfa_land_share_by_tract.csv` | FHFA's land share of single-family value, by tract | FHFA land-price dataset (Davis, Larson, Oliner and Shui), "Cross-Section Census Tracts" sheet, moved from 2010 to 2020 tracts by `scripts/build_philadelphia_fhfa_tract_shares.py` |
| `cities/philadelphia/data/census_tracts.gpq`, `council_districts.gpq` | map outlines | Census TIGERweb; OpenDataPhilly |
| `analysis/data/philadelphia_abatement_phase_in_*` | the year-by-year abatement transition | `scripts/philadelphia_abatement_phase_in.py`, from each abated parcel's exemption history in OPA's `assessments` table |

In [ ]:
inputs = {
    "parcel roll": op.CACHE_DIR / f"parcels_ty{op.TAX_YEAR}.gpq",
    "S5 land values": op.SURFACE_EXPORT,
    "S5 support edge": op.SURFACE_SUPPORT,
    "L&I vacant land": op.VACANT_LNI,
    "OPA surface parking": op.SURFACE_PARKING,
    "OPA attributes": op.OPA_ATTRIBUTES,
    "OPA mailing addresses": REPO / "analysis/ownership/philadelphia/opa_mailing.parquet",
    "FHFA tract land shares": op.FHFA_TRACTS,
    "census tracts": op.CACHE_DIR / "census_tracts.gpq",
    "council districts": op.COUNCIL_DISTRICTS,
}
pd.DataFrame([dict(input=k, path=str(v.relative_to(REPO)).replace("\\", "/"), exists=v.exists(),
                   MB=round(v.stat().st_size / 1e6, 1) if v.exists() else None,
                   file_date=op.file_date(v) if v.exists() else None)
              for k, v in inputs.items()]).set_index("input")

### Rebuilding the inputs from scratch

Run from the repository root, in this order. Nothing here is needed to run the rest of the notebook.

1. **Land values from land sales** (separate repository, `philly_open_avmkit`). Its pipeline notebooks
   `notebooks/pipeline/00-download.ipynb` → `01-assemble` → `02-clean` → `03-model` download and clean
   OPA properties, sales and the Department of Records parcel map; `notebooks/pipeline/run_land_surfaces.py`
   fits the land surfaces (about 40 minutes each) and `build_land_roll.py` turns S5 into a land roll.
2. **The parcel roll:** `python scripts/build_philadelphia_parcel_cache.py --year 2026 --force`
3. **S5 onto parcels, plus demographics:** run `cities/philadelphia/model_lycd_reassessment.ipynb` with
   the environment variable `LVT_LAND_SURFACE=s5`.
4. **Vacant land and parking:** `python scripts/fetch_philly_vacant_parking.py`
5. **OPA attributes and mailing addresses:** `python analysis/ownership/philadelphia/fetch_opa_attributes.py`
   and `python analysis/ownership/philadelphia/fetch_mailing.py`
6. **Council districts:** `python scripts/map_philadelphia_tax_changes.py` (caches them on first run)
7. **Abatements:** `python scripts/build_philadelphia_abatement_classification.py`, then
   `python scripts/philadelphia_abatement_phase_in.py --homestead-order value_share --baseline rate --revalue-bare-lots`
8. **The flyer's numbers:** `python scripts/philadelphia_council_one_pager.py --homestead-order value_share`
   writes `numbers.json` and the map; section 15 checks this notebook against it.

## 2. Tax parameters for tax year 2026

The rate is the City's plus the School District's, and the Homestead Exemption takes the first
$100,000 of an owner-occupied home's assessment off the bill. Both are cited to City sources in
`lvt/philadelphia.py`.

In [ ]:
params = tax_year_params(op.TAX_YEAR)
print(params.describe() if hasattr(params, "describe") else params)
print(f"\ncombined rate: {params.combined_mills / 10:.4f}%   Homestead Exemption: ${params.homestead_exemption:,}")
print("source:", params.source)

## 3. The parcel roll, with sales-based land values joined on

`load_parcels` reads OPA's tax year 2026 roll (one row per parcel: taxable and exempt land and
building, the Homestead Exemption, OPA's category code, lot area, owner) and joins three things to
it by parcel number:

- **`s5_land`**, the land value estimated from land sales (the "S5" surface, section 4);
- **`land_beyond_support`**, whether the lot is bigger than the land sales can vouch for;
- the parcel's **census block group** income and non-white share (for section 9).

A handful of parcel numbers name more than one physical parcel; those are dropped rather than matched
arbitrarily.

In [ ]:
g = op.load_parcels()
g[["parcel_number", "category_code", "taxable_land", "exempt_land", "taxable_building", "exempt_building",
   "homestead_exemption", "s5_land", "land_beyond_support", "total_area", "median_income", "minority_pct"]].head()

## 4. Property types, and the world after today's abatements

`classify` turns OPA's category codes into property types, with the overrides documented in
`cities/philadelphia/CLAUDE.md`: a parcel with no taxable building is vacant land, an abated parcel
(whose new building is temporarily off the roll) is recognised as abated rather than vacant, and a
fully exempt parcel is marked exempt.

The flyer describes the city **after today's 10-year abatements have run out** (the last expires in
2036). So `expire_abatements` puts each abated building back on the taxable roll. Comparing the
reform against a roll that still carries today's abatements would credit the land value tax with
revenue that current law collects anyway a few years later. The transition while abatements are
still running is section 13.

In [ ]:
category, abated, restored_building, full_exempt = op.classify(g)
post = op.expire_abatements(g, abated, restored_building)
taxable = ~full_exempt
homes = category.isin(op.HOMES).to_numpy() & taxable
groups = category.map(op.property_group)
print(f"\n{taxable.sum():,} parcels with some taxable value; {homes.sum():,} of them homes (1-4 units); "
      f"{abated.sum():,} abated today")
groups[taxable].value_counts().to_frame("taxable parcels")

### OPA's land values against the land-sales estimate

The reform depends on the land/building split, so it depends on the land values. OPA does not value
land from land sales; its land values are closer to a fixed share of each property's value. The
S5 surface is estimated from Philadelphia's own sales of vacant land, adjusted for location and lot
size, and cross-checked on sales held out of the fit. Concretely, S5 is a *paired-sales* surface: it takes sales of bare lots (and teardowns) near one another in the same zoning family, and compares them one attribute at a time, holding location fixed, to learn how price per square foot changes with lot size. Each parcel is then valued from its 8 nearest comparable land sales, adjusted to its own lot size. Where there is no nearby sale it takes its neighbors' rate. It lives in the `philly_open_avmkit` repository.

The chart shows what that does to single-family homes. OPA puts almost every one of them at the same
land share, about a fifth of its value, whatever the neighborhood. The land sales say land is a much
larger share of value in some neighborhoods and a much smaller one in others. That spread is what
makes the reform progressive: where land is cheap relative to the house, the land share is low and
those homes save. (The small bar at 100% is homes whose sales-based land value exceeds OPA's whole
assessment; their land is capped at that total, since the total is held fixed.)

In [ ]:
sfr = (category == "Single Family Residential").to_numpy() & taxable
gross_land_opa = (post.taxable_land + post.exempt_land).to_numpy()
gross_total = gross_land_opa + (post.taxable_building + post.exempt_building).to_numpy()
ok = sfr & (gross_total > 0)
opa_share = 100 * gross_land_opa[ok] / gross_total[ok]
s5_share = 100 * np.minimum(post.s5_land.to_numpy()[ok], gross_total[ok]) / gross_total[ok]

fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(0, 100, 51)
ax.hist(opa_share, bins=bins, color=MUTED, alpha=.55, label="OPA's land value")
ax.hist(s5_share, bins=bins, color=PURPLE, alpha=.6, label="land-sales estimate (S5)")
ax.annotate("OPA: almost every home\nat the same land share", xy=(np.median(opa_share) + 1, ax.get_ylim()[1] * .8),
            xytext=(45, ax.get_ylim()[1] * .6), fontsize=9, arrowprops=dict(arrowstyle="->", color=MUTED))
ax.set(xlabel="land as % of the home's assessed value", ylabel="single-family homes",
       title="Single-family homes: land share of value")
ax.legend(frameon=False)
plt.show()
print(f"median land share: OPA {np.median(opa_share):.0f}%, S5 {np.median(s5_share):.0f}%")

## 5. The reform: re-split each total, then solve the two rates

`shift` does three things:

1. **Re-split, holding the total.** For a parcel with a building, land becomes the S5 estimate
   (capped at the parcel's total) and the building is the rest: `land = min(S5, total)`,
   `building = total − land`. The assessment itself does not move, so no building is revalued and
   nobody's total goes up. This is what a Council ordinance setting land-assessment standards would
   do (step 1 on the flyer's back).
2. **Bare lots are valued at their land estimate**, since a lot with no building has nothing to hold
   a total against. Lots bigger than the land sales can test keep OPA's value instead (section 14).
3. **Solve the rates.** With land taxed at 4 times the building rate, the one rate that raises
   today's revenue is

   $$\text{building rate} = \frac{\text{revenue}}{4 \times \text{taxable land} + \text{taxable building}},
   \qquad \text{land rate} = 4 \times \text{building rate}.$$

In [ ]:
headline = op.shift(post, taxable, params, ORDER)
land, building, current, new = headline.land, headline.building, headline.current, headline.new
print(f"revenue today:         ${headline.revenue / 1e9:,.3f} billion")
print(f"taxable land:          ${land.sum() / 1e9:,.1f} billion")
print(f"taxable building:      ${building.sum() / 1e9:,.1f} billion")
print(f"building rate:         {headline.building_mills / 10:.4f}%   (today {params.combined_mills / 10:.4f}%)")
print(f"land rate:             {headline.land_mills / 10:.4f}%")
print(f"revenue after:         ${new.sum() / 1e9:,.3f} billion  (revenue-neutral: {abs(new.sum() / headline.revenue - 1) < 1e-9})")

built = taxable & ~headline.bare
same_total = np.isclose(land + building, headline.base_total, atol=1.0)
print(f"\n{built.sum():,} taxable parcels with a building; total assessment unchanged on {same_total[built].mean():.1%}"
      f"\n(the rest carry relief, like the Homestead Exemption, whose value depends on the split)")
print(f"{headline.bare.sum():,} bare lots valued at their land estimate")

The flyer prints these two rates rounded to two decimals.

**Why a property pays less when its land is under about a third of its value.** With the total held
fixed, a property's bill changes in proportion to (its land share − the city's taxable land share).
The city's taxable land share is printed below; the flyer rounds it to "about a third", and a
property with less land than that pays less. That is also why the ratio (4:1) sets how strongly the reform bites, not who wins.

In [ ]:
city_land_share = 100 * land.sum() / (land.sum() + building.sum())
print(f"citywide taxable land share: {city_land_share:.1f}%")

## 6. The Homestead Exemption under two rates

At one rate it does not matter which part of a home the $100,000 exemption comes off. Under two
rates it matters a great deal. Current state law, **53 Pa.C.S. § 8583(c)**, takes it off the building
first. Under a split rate that makes the exemption worth $100,000 × the *building* rate, about half
what it is worth today, so most homestead owners would pay more even though the reform cuts their
building's rate.

The flyer instead takes the exemption off land and building **in proportion to their value**
(`value_share`), which keeps it worth about what it is worth today. That needs § 8583(c) amended,
which is why the flyer's step 2 says the enabling bill must also fix the Homestead Exemption. The
four orders the library can model:

In [ ]:
for k in HOMESTEAD_ORDERS:
    print(f"{k:15s} {op.HOMESTEAD_LABELS[k]}")

## 7. Who saves

"Homes" are taxable 1–4-unit residential parcels with a tax bill today. The flyer's headline counts
the share whose bill falls, and the median change in dollars.

In [ ]:
h = op.bill_stats(current, new, homes)
hs = _decompose_exemptions(post, "homestead_exemption", 1.0).homestead_active.to_numpy()
hstats = op.cohort_stats(current, new, homes & hs)
print(f"homes paying less:      {h['pay_less_pct']:.1f}%  -> '{round(h['pay_less_pct'] / 10)} in 10'")
print(f"median change:          ${h['median_change_usd']:,.0f} on a median bill of ${h['median_current_bill_usd']:,.0f}")
print(f"homestead homes:        {hstats['pay_less_pct']:.1f}% pay less (median saving ${-hstats['median_change_usd']:,.0f}); "
      f"{hstats['pay_more_pct']:.1f}% pay more (median ${hstats['median_increase_of_those_paying_more_usd']:,.0f})")

In [ ]:
d = (new - current)[homes & (current > 0)]
fig, ax = plt.subplots(figsize=(7, 3))
bins = np.arange(-1500, 1501, 50)
clipped = np.clip(d, -1500, 1500)
ax.hist(clipped[clipped < 0], bins=bins, color=PURPLE)
ax.hist(clipped[clipped >= 0], bins=bins, color=RED)
ax.axvline(0, color=INK, lw=.8)
ax.set(xlabel="change in yearly tax bill, $ (clipped at ±$1,500)", ylabel="homes",
       title="Homes: change in yearly bill")
plt.show()

## 8. The rowhouse example

The flyer's example house is built from the **median land and median building** of single-family
homes with a Homestead Exemption, not the house at the median bill (those are different houses). Its
twin is the same house without the exemption (a rental), and the empty lot next door has the same
land and no building. Each bill goes through the same library rule as the whole roll.

In [ ]:
t = op.typical_rowhouse(headline, homes, category, params, ORDER)
print(f"typical homestead home: land ${t['land_usd']:,.0f}, building ${t['building_usd']:,.0f} "
      f"(land {t['land_share_pct']:.0f}% of value)")
for k, label in (("with_homestead", "owner's rowhouse"), ("without_homestead", "same house, rented"),
                 ("empty_lot", "empty lot, same land")):
    v = t[k]
    print(f"  {label:22s} ${v['today_usd']:,.0f} -> ${v['after_usd']:,.0f}  ({v['change_usd']:+,.0f}, {v['change_pct']:+.0f}%)")
print(f"pays less when land is under {t['with_homestead']['pays_less_below_land_share_pct']:.0f}% of value")

The same house at every possible land share shows where the break-even comes from:

In [ ]:
value = t["value_usd"]
shares = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(7, 3.2))
for hs_flag, label, color in ((True, "with Homestead Exemption", PURPLE), (False, "without", MUTED)):
    today, after = op.house_bills(shares * value, (1 - shares) * value, hs_flag, ORDER, params,
                                  headline.land_mills, headline.building_mills)
    ax.plot(100 * shares, after - today, color=color, label=label)
ax.axhline(0, color=INK, lw=.8)
ax.axvline(t["with_homestead"]["pays_less_below_land_share_pct"], color=AMBER, lw=2, ls="--")
ax.axvline(t["land_share_pct"], color=INK, lw=.8, ls=":")
ax.set(xlabel=f"land as % of a ${value:,.0f} home's value", ylabel="change in yearly bill, $",
       title="Same total value, different land share")
ax.legend(frameon=False)
plt.show()

## 9. Neighborhoods: the biggest cuts go to poorer and more diverse areas

Every home is placed in its census block group, and block groups are ranked into fifths by median
household income and by non-white share of residents (fifths are cut so each holds a fifth of the
city's homes). For each fifth: the change in homes' **total** tax bill, and the share of homes paying
less. These are neighborhood averages, not households.

In [ ]:
q, edges = op.strata(g, homes)
profile = op.neighbourhood_profile(q, current, new, homes)
rows = []
for col, title in (("inc_q", "income"), ("min_q", "non-white share")):
    for k, v in profile[col].items():
        rows.append(dict(ranked_by=title, fifth=k, homes_bill_change_pct=v["homes_pct"], pay_less_pct=v["pay_less_pct"],
                         median_change_usd=v["median_change_usd"]))
pd.DataFrame(rows).set_index(["ranked_by", "fifth"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), sharex=True)
for ax, (col, title, order) in zip(axes, (
        ("inc_q", "By median household income", ["poorest", "Q2", "Q3", "Q4", "richest"]),
        ("min_q", "By non-white share of residents", ["most non-white", "Q4", "Q3", "Q2", "whitest"]))):
    vals = [profile[col][k]["homes_pct"] for k in order]
    ax.barh(range(5), vals, color=[PURPLE if v < 0 else RED for v in vals])
    ax.set_yticks(range(5), order)
    ax.invert_yaxis()
    ax.axvline(0, color=INK, lw=.8)
    for i, v in enumerate(vals):
        ax.text(v + (0.6 if v >= 0 else -0.6), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("change in homes' total bill, %")
    ax.set_xlim(min(vals) - 7, max(max(vals), 0) + 5)
plt.tight_layout()
plt.show()
print(f"poorest fifth: {profile['inc_q']['poorest']['pay_less_pct']:.0f}% of homes pay less; "
      f"most non-white fifth: {profile['min_q']['most non-white']['pay_less_pct']:.0f}%")

### Does this depend on the land-sales estimate? An independent check

The pattern could be an artefact of the S5 land values. So the whole calculation is re-run with a
completely separate source: the Federal Housing Finance Agency's estimates of land's share of
single-family property value, by census tract. The cell below compares the gap between the poorest
and richest fifths (and between the most and least non-white) under each source. The flyer's
"a little over half the size" describes the ratio it prints.

In [ ]:
rob = op.progressivity_robustness(g, post, category, taxable, homes, q, params, ORDER, profile)
for col, v in rob["spread"].items():
    print(f"{v['compare']}: S5 {v['s5']:+.1f} points, FHFA {v['fhfa']:+.1f} points -> FHFA/S5 = {v['fhfa_over_s5']:.2f}")
print(f"(single-family homes matched to an FHFA tract: {rob['basis']['single_family_matched_pct']:.0f}%)")

## 10. Property types: building-heavy pays less, land-heavy pays more

The share of each group's properties whose bill falls. Commercial property is mixed: downtown towers
and hotels (mostly building) pay less, while warehouses, gas stations, strip retail and parking lots
(mostly land) pay more.

In [ ]:
by_group = op.sector_totals(groups, taxable, current, new)
by_group[["parcels", "pay_less_pct", "change_pct", "change_musd", "median_change_pct"]]

In [ ]:
order_ = ["homes", "large multifamily", "commercial/mixed/other", "vacant land"]
labels = ["Homes", "Apartments", "Commercial", "Vacant lots"]
less = by_group.loc[order_, "pay_less_pct"].to_numpy()
fig, ax = plt.subplots(figsize=(6, 2.2))
ax.barh(labels, less, color=PURPLE)
ax.barh(labels, 100 - less, left=less, color=RED)
ax.invert_yaxis()
ax.set(xlim=(0, 100), xlabel="% of properties paying less (purple) or more (red)")
plt.show()

## 11. Idle land the tax can reach

Two city records define it: the Department of Licenses and Inspections' vacant-land determinations,
and OPA's building codes for surface parking lots. Only parcels with **taxable** value count: public,
Land Bank and other exempt land is left out, because a tax rate cannot reach it. A parcel in both
layers counts once, as parking. Areas are OPA's recorded lot areas.

In [ ]:
idle = op.idle_land(g, taxable, current, new)
rows = {k: idle[k] for k in ("vacant", "parking", "total")}
display(pd.DataFrame(rows).T[["parcels", "acres", "pay_more_pct", "exempt_parcels", "exempt_acres"]])
print(f"{idle['total']['acres']:,.0f} taxable acres = {idle['pct_of_city_land']:.1f}% of the city's land; "
      f"a square {idle['square_side_mi']:.1f} miles on a side; {idle['central_parks']:.2f} times Central Park")

In [ ]:
# The flyer's map: draw it into a temporary folder so the canonical copy is not touched
op.OUT = Path(tempfile.mkdtemp())
op.render_map(idle)
%matplotlib inline
display(Image(op.OUT / "gathered_light.png", width=320))

## 12. The Homestead Exemption under current law

The same reform under each Homestead Exemption order, with the rates re-solved each time. Under the
current statute (`building_first`) most homestead owners pay more; that is the figure on the flyer's
step 2. This runs the reform four times, so it takes a minute.

In [ ]:
occupied = hs | op.owner_occupied_proxy(g, hs, abated)
hc = op.homestead_comparison(post, category, taxable, homes, groups, q, params, hs, occupied)
pd.DataFrame({k: dict(rule=v["rule"], land_rate_pct=v["land_rate_pct"], building_rate_pct=v["building_rate_pct"],
                      homestead_pay_more_pct=v["homes"]["homestead"]["pay_more_pct"],
                      all_homes_pay_less_pct=v["homes"]["all_homes"]["pay_less_pct"],
                      exemption_worth_after_usd=v["exemption_worth_usd"]["median_after"])
              for k, v in hc["orders"].items()}).T

## 13. The 10-year abatement: the transition

No abatement ends early. `scripts/philadelphia_abatement_phase_in.py` models the reform year by year,
phasing the ratio up to 4:1 over 10 years while each abatement runs out on its own schedule, and
compares every abated property's bills against current law. `payback` reads that run (and checks its
final-year rates against section 5's) and sorts abated properties three ways:

- never behind: never pays more in total;
- behind, then made whole: pays more while the abatement runs, and recoups it after;
- never recoups: pays more for good (land-heavy properties).

In [ ]:
tr = op.payback(ORDER, headline)
parts = [("never pay more", tr["never_behind_pct"], PURPLE), ("pay more for a time", tr["behind_then_recoups_pct"], AMBER),
         ("pay more for good", tr["never_recoups_pct"], RED)]
fig, ax = plt.subplots(figsize=(7, 1.4))
left = 0
for label, v, color in parts:
    ax.barh(0, v, left=left, color=color, height=.5)
    ax.text(left, -.45, f"{v:.0f}% {label}", ha="left", va="top", fontsize=8)
    left += v
ax.set(xlim=(0, 100), ylim=(-1, .4))
ax.set_axis_off()
plt.show()
print(f"{tr['abated_parcels']:,} abated properties. Of those made whole, half recoup within "
      f"{tr['recoup_years_after_expiry_median']:.1f} years of expiry, nine in ten within {tr['recoup_years_after_expiry_p90']:.0f}.")
print(f"once abatements end, {tr['after_expiry_pay_less_pct']:.0f}% pay less than under today's tax, "
      f"a median ${-tr['after_expiry_median_annual_usd']:,.0f} a year; the last expires in {tr['last_expiry_tax_year']}")

## 14. The largest empty tracts, and why homes' total is a range

The land sales that S5 is fitted on are mostly small lots, so beyond a certain lot size it is
extrapolating. For bare lots larger than that (counted below), three readings are defensible: keep OPA's
value (the headline), re-level OPA's value by how far it runs below the sales that do exist, or take
S5 whole. They move a lot of land value, so the dollar total for homes is printed as a range. The
shares of properties paying less barely move.

In [ ]:
beyond = headline.bare & post.land_beyond_support.to_numpy()
print(f"{beyond.sum():,} bare lots are larger than the land sales can test")
rows = {}
for treatment in LARGE_TRACT_TREATMENTS:
    alt = headline if treatment == "carry_opa" else op.shift(post, taxable, params, ORDER, large_tracts=treatment)
    s = op.sector_totals(groups, taxable, alt.current, alt.new)
    rows[treatment] = dict(land_rate_pct=alt.land_mills / 10, building_rate_pct=alt.building_mills / 10,
                           homes_change_musd=s.loc["homes", "change_musd"], homes_pay_less_pct=s.loc["homes", "pay_less_pct"])
rng = pd.DataFrame(rows).T
display(rng)
print(f"homes as a whole pay ${-rng.homes_change_musd.max():,.0f} million to ${-rng.homes_change_musd.min():,.0f} million less a year")

## 15. Check against the flyer's numbers

Every value computed above, next to the same value in `numbers.json`, the file the flyer's figures are
taken from. A mismatch means one of them was produced from different data or code, and the flyer should not be
trusted until they agree.

In [ ]:
numbers_path = REPO / "analysis/political/philadelphia_council_one_pager/numbers.json"
if not numbers_path.exists():
    print(f"{numbers_path} not found: run scripts/philadelphia_council_one_pager.py --homestead-order value_share")
else:
    n = json.loads(numbers_path.read_text(encoding="utf-8"))
    vs = n["homestead_comparison"]["orders"]["value_share"]["homes"]["homestead"]
    bf = n["homestead_comparison"]["orders"]["building_first"]["homes"]["homestead"]
    checks = [
        ("land rate %", headline.land_mills / 10, n["rates"]["land_rate_pct"]),
        ("building rate %", headline.building_mills / 10, n["rates"]["building_rate_pct"]),
        ("homes paying less %", h["pay_less_pct"], n["homes"]["pay_less_pct"]),
        ("median home change $", h["median_change_usd"], n["homes"]["median_change_usd"]),
        ("homestead paying less %", hstats["pay_less_pct"], vs["pay_less_pct"]),
        ("homestead median increase $", hstats["median_increase_of_those_paying_more_usd"], vs["median_increase_of_those_paying_more_usd"]),
        ("rowhouse after $", t["with_homestead"]["after_usd"], n["typical_rowhouse"]["with_homestead"]["after_usd"]),
        ("empty lot after $", t["empty_lot"]["after_usd"], n["typical_rowhouse"]["empty_lot"]["after_usd"]),
        ("poorest fifth change %", profile["inc_q"]["poorest"]["homes_pct"], n["homes_profile"]["inc_q"]["poorest"]["homes_pct"]),
        ("most non-white fifth change %", profile["min_q"]["most non-white"]["homes_pct"], n["homes_profile"]["min_q"]["most non-white"]["homes_pct"]),
        ("FHFA / S5, income", rob["spread"]["inc_q"]["fhfa_over_s5"], n["progressivity_robustness"]["spread"]["inc_q"]["fhfa_over_s5"]),
        ("commercial paying less %", by_group.loc["commercial/mixed/other", "pay_less_pct"], n["by_property_group"]["commercial/mixed/other"]["pay_less_pct"]),
        ("idle taxable acres", idle["total"]["acres"], n["idle_land"]["total"]["acres"]),
        ("idle paying more %", idle["total"]["pay_more_pct"], n["idle_land"]["total"]["pay_more_pct"]),
        ("homestead paying more, current law %", hc["orders"]["building_first"]["homes"]["homestead"]["pay_more_pct"], bf["pay_more_pct"]),
        ("abated never behind %", tr["never_behind_pct"], n["transition"]["never_behind_pct"]),
        ("abated after-expiry median $", tr["after_expiry_median_annual_usd"], n["transition"]["after_expiry_median_annual_usd"]),
        ("homes $M, surface reading", rows["surface"]["homes_change_musd"], n["large_tract_range"]["surface"]["by_property_group"]["homes"]["change_musd"]),
    ]
    table = pd.DataFrame(checks, columns=["figure", "this notebook", "numbers.json"]).set_index("figure")
    table["match"] = np.isclose(table["this notebook"], table["numbers.json"], rtol=1e-6, atol=0.01)
    display(table)
    print(f"numbers.json as of {n['as_of']}: {table.match.sum()} of {len(table)} match")
    assert table.match.all(), "the notebook and numbers.json disagree; regenerate one of them"